<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/notebook-02-tokenizer-mechanics/notebooks/02_tokenizer_training_and_corpus_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 02 — Tokenizer Training & Corpus Construction

This notebook begins from the verified data contract established in **Notebook 01 — Data Preparation & Corpus Audit**. It independently reloads the immutable WikiText-103 revision, applies the same locked normalization and article-reconstruction logic through `src/data.py`, trains the project tokenizer from scratch, validates it, records its checksum, and constructs the exact 20,000,000-token model-training corpus.

Notebook 01 is a completed audit artifact. Notebook 02 must not depend on Notebook 01's in-memory state.

### Pipeline boundary

```text
Notebook 01: raw WikiText → verified normalized articles
src/data.py: canonical reusable loading/normalization/reconstruction logic
Notebook 02: normalized articles → tokenizer → exact 20M-token corpus
Notebook 03: tokenizer/corpus → Transformer architecture
```


### Locked inputs and constraints

- Dataset: `Salesforce/wikitext`, `wikitext-103-raw-v1`
- Immutable Hub revision: `b08601e04326c79dfdd32d625aee71d232d685c3`
- Tokenizer: byte-level BPE trained from scratch
- Vocabulary size: 16,384 total tokens, including registered special tokens
- Tokenizer-training text: full normalized official training split only
- Model-training corpus: exactly 20,000,000 tokenizer-produced tokens
- Sampling seed: 42
- Validation remains development-visible; test remains untouched until final evaluation
- Article boundaries and normalization must reproduce Notebook 01's verified 28,472 training documents and 60 validation documents before tokenizer work proceeds

Canonical references: `docs/PROJECT_CONTEXT.md` and `docs/DECISION_REGISTER.md`.

# Chunk 1 — Reproduce the audited corpus from shared source code

Before tokenizer design begins, this chunk proves that Notebook 02 can independently reproduce Notebook 01's audited corpus. The verified normalization and article-reconstruction implementation has been extracted into `src/data.py` without refactoring the core logic.

The notebook pins the **source-code revision** containing that extraction. This matters because the Hub dataset revision alone fixes the upstream text, while the source-code revision fixes the exact transformation applied to it.

## 1. Load the canonical data pipeline at a fixed source revision

A Colab notebook opened from GitHub does not automatically make the repository's `src/` package importable. We therefore clone the project repository, check out the exact commit that introduced the verified extraction, and add the repository root to Python's import path.

This is intentionally a source-code pin, not a dependency install. The goal is for a fresh runtime to reconstruct the same data pipeline without relying on Notebook 01 or on whatever happens to be at the tip of `main` later.

In [ ]:
%pip install -q datasets "tokenizers==0.23.1"


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/traderjohnd/foundation-model-from-scratch.git"
REPO_DIR = Path("/content/foundation-model-from-scratch")
DATA_PIPELINE_REVISION = "7d300f14c812d9a1caf36aa9ec0568bee5b0f275"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", DATA_PIPELINE_REVISION], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Loaded project source revision: {DATA_PIPELINE_REVISION}")


Loaded project source revision: 7d300f14c812d9a1caf36aa9ec0568bee5b0f275


In [ ]:
import pandas as pd

from src.data import (
    AUDIT_SPLITS,
    DATASET_CONFIG,
    DATASET_ID,
    DATASET_REVISION,
    EXPECTED_ARTICLE_COUNTS,
    load_pinned_wikitext,
    normalize_development_splits,
    reconstruct_development_articles,
    run_normalization_self_test,
)

pd.Series({
    "dataset": DATASET_ID,
    "config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "source_revision": DATA_PIPELINE_REVISION,
    "development_splits": AUDIT_SPLITS,
})


,0
dataset,Salesforce/wikitext
config,wikitext-103-raw-v1
dataset_revision,b08601e04326c79dfdd32d625aee71d232d685c3
source_revision,7d300f14c812d9a1caf36aa9ec0568bee5b0f275
development_splits,"(train, validation)"


## 2. Re-run the locked normalization regression tests

Notebook 01 established 17 explicit normalization cases. Running the same cases through `src/data.py` checks that extraction did not silently change the transformation contract.

In [ ]:
normalization_test_results = run_normalization_self_test()
normalization_tests = pd.DataFrame(normalization_test_results)

assert len(normalization_tests) == 17
assert normalization_tests["passed"].all()
print("✓ 17/17 normalization regression tests passed.")
normalization_tests


✓ 17/17 normalization regression tests passed.


,input,expected,actual,passed
0,well @-@ known,well-known,well-known,True
1,3 @.@ 5 million,3.5 million,3.5 million,True
2,"1 @,@ 000 people","1,000 people","1,000 people",True
3,( 1987 ),(1987),(1987),True
4,don 't,don't,don't,True
5,café 's tables,café's tables,café's tables,True
6,the players ' hopes,the players ' hopes,the players ' hopes,True
7,$ 3 @.@ 5 million,$3.5 million,$3.5 million,True
8,The meeting ran from 12 : 30 to 13 : 05 .,The meeting ran from 12:30 to 13:05.,The meeting ran from 12:30 to 13:05.,True
9,"the "" Nameless "", a penal unit","the ""Nameless"", a penal unit","the ""Nameless"", a penal unit",True


## 3. Reload the immutable WikiText-103 revision

The module loads the same pinned Hub commit used by Notebook 01 and hard-checks the official row counts. Loading the dataset does **not** authorize development-time inspection of test examples; only `train` and `validation` are transformed below.

In [ ]:
raw_dataset = load_pinned_wikitext()

split_rows = pd.Series(
    {split_name: raw_dataset[split_name].num_rows for split_name in raw_dataset},
    name="rows",
)
split_rows


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

,rows
test,4358
train,1801350
validation,3760


## 4. Normalize development-visible splits only

The same locked function is applied to the official training and validation splits. The test split is deliberately not normalized or inspected during development.

In [ ]:
normalized_development = normalize_development_splits(raw_dataset)

assert set(normalized_development) == {"train", "validation"}
print("✓ Normalized only train and validation.")


Normalize train:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Normalize validation:   0%|          | 0/3760 [00:00<?, ? examples/s]

✓ Normalized only train and validation.


## 5. Reconstruct articles and enforce the audit contract

Article starts are detected from the **raw** rows using the locked level-1-heading-plus-blank-neighbors rule, while the stored article text comes from the normalized rows. This is the same distinction that resolved the boundary-count discrepancies in Notebook 01.

The hard assertions below are the handoff gate. Tokenizer work does not proceed unless the shared module independently reproduces **28,472 training documents and 60 validation documents**.

In [ ]:
articles_by_split = reconstruct_development_articles(
    raw_dataset,
    normalized_development,
)

article_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles": len(articles),
            "expected": EXPECTED_ARTICLE_COUNTS[split_name],
            "characters": sum(len(article["text"]) for article in articles),
            "first_article_id": articles[0]["article_id"],
            "last_article_id": articles[-1]["article_id"],
        }
        for split_name, articles in articles_by_split.items()
    ]
).set_index("split")

assert article_summary.loc["train", "articles"] == 28_472
assert article_summary.loc["validation", "articles"] == 60
print("✓ Shared pipeline reproduced the audited article counts.")
article_summary


✓ Shared pipeline reproduced the audited article counts.


,articles,expected,characters,first_article_id,last_article_id
split,,,,,
train,28472,28472,519078250,train:article-00000:row-1,train:article-28471:row-1801326
validation,60,60,1102768,validation:article-00000:row-1,validation:article-00059:row-3714


### Chunk 1 checkpoint — passed

A fresh runtime reloaded the pinned upstream corpus, executed the canonical shared preprocessing implementation, reran all 17 normalization tests, and reproduced the audited **28,472 training / 60 validation** article counts. Notebook 02 can therefore proceed to tokenizer design without depending on Notebook 01 kernel state.

# Chunk 2 — Lock the document-boundary / EOS special-token contract

Before BPE training begins, we need to decide which symbols are **structural** rather than learned from ordinary text. The project will use **`<|endoftext|>`** as the single registered special token. It serves as the document-boundary / end-of-sequence marker that will later be appended explicitly after each complete article.

This is a vocabulary contract, not tokenizer training. No BPE merges are learned in this chunk.

## 6. Define the structural vocabulary contract

The total vocabulary remains **16,384 IDs**, and the special token is reserved **inside** that total. We register no PAD, BOS, or UNK token.

- **No PAD:** the training pipeline will use fixed-length token sequences rather than padded variable-length examples.
- **No BOS:** the project does not need a separate beginning-of-sequence marker; document separation is handled by the boundary/EOS token.
- **No UNK:** byte-level BPE is designed to retain byte coverage, so an unknown-token fallback is unnecessary. Byte coverage will be explicitly validated after tokenizer training.
- **One boundary/EOS token:** `<|endoftext|>` is inserted by corpus construction, not automatically by ordinary text encoding.

Because the byte-level alphabet contains 256 byte symbols, a final 16,384-token vocabulary that reaches its requested size would contain 1 special token, 256 byte-alphabet tokens, and up to 16,127 merge-created vocabulary entries. We will verify the actual trained vocabulary rather than assuming the trainer reaches this exact composition.

In [ ]:
VOCAB_SIZE = 16_384
DOC_BOUNDARY_TOKEN = "<|endoftext|>"
SPECIAL_TOKENS = [DOC_BOUNDARY_TOKEN]
BYTE_ALPHABET_SIZE = 256

assert len(SPECIAL_TOKENS) == 1
assert VOCAB_SIZE > BYTE_ALPHABET_SIZE + len(SPECIAL_TOKENS)

vocab_contract = pd.Series({
    "total_vocab_size": VOCAB_SIZE,
    "registered_special_tokens": len(SPECIAL_TOKENS),
    "document_boundary_token": DOC_BOUNDARY_TOKEN,
    "byte_alphabet_size": BYTE_ALPHABET_SIZE,
    "non_special_vocab_slots": VOCAB_SIZE - len(SPECIAL_TOKENS),
    "max_merge_created_slots_if_full": (
        VOCAB_SIZE - len(SPECIAL_TOKENS) - BYTE_ALPHABET_SIZE
    ),
})

vocab_contract


,0
total_vocab_size,16384
registered_special_tokens,1
document_boundary_token,<|endoftext|>
byte_alphabet_size,256
non_special_vocab_slots,16383
max_merge_created_slots_if_full,16127


## 7. Prove the boundary-token string does not occur naturally

A registered special token must have an unambiguous structural meaning. If the literal string `<|endoftext|>` already appeared inside an article, that natural text could be confused with the boundary marker once the tokenizer treats the string as a special symbol.

We therefore search only the development-visible reconstructed `train` and `validation` articles. The test split remains uninspected.

In [ ]:
boundary_collision_counts = {}

for split_name in AUDIT_SPLITS:
    literal_matches = sum(
        DOC_BOUNDARY_TOKEN in article["text"]
        for article in articles_by_split[split_name]
    )
    boundary_collision_counts[split_name] = literal_matches
    assert literal_matches == 0, (
        f"{DOC_BOUNDARY_TOKEN!r} occurs literally in {split_name}: "
        f"{literal_matches} articles"
    )

print("✓ Boundary-token literal collision check passed.")
pd.Series(boundary_collision_counts, name="articles_with_literal_boundary_token")


✓ Boundary-token literal collision check passed.


,articles_with_literal_boundary_token
train,0
validation,0


## 8. Keep literal `<unk>` as ordinary corpus text

WikiText can contain the literal characters `<unk>` as part of the released corpus. We **do not** register `<unk>` as a tokenizer special token. Doing so would change the meaning of those existing corpus occurrences from ordinary text into a control symbol.

Instead, we simply measure how often the literal string appears in development-visible articles. This is an observation, not an assertion: the observed counts are recorded after execution.

In [ ]:
LITERAL_UNK = "<unk>"

unk_inventory = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles_with_literal_unk": sum(
                LITERAL_UNK in article["text"]
                for article in articles_by_split[split_name]
            ),
            "literal_unk_occurrences": sum(
                article["text"].count(LITERAL_UNK)
                for article in articles_by_split[split_name]
            ),
        }
        for split_name in AUDIT_SPLITS
    ]
).set_index("split")

unk_inventory


,articles_with_literal_unk,literal_unk_occurrences
split,,
train,0,0
validation,0,0


### Pause here

This chunk locks the special-token contract and checks the corpus against it. **Do not train the tokenizer yet.**

Execution gate for the next chunk:
- the boundary-token collision count must be zero for both train and validation;
- the observed literal `<unk>` counts should be recorded without turning `<unk>` into a special token; and
- the 16,384-token vocabulary accounting must remain explicit.

**Next reviewed chunk:** configure the actual byte-level BPE tokenizer and verify the special-token ID / byte-level mechanics before full tokenizer training.

# Chunk 3 — Probe the byte-level BPE mechanics

Before training on the full ~519M-character WikiText training corpus, we will build a **tiny probe tokenizer** using the structural settings already locked for the project.

This probe is deliberately small. It answers four mechanical questions:

1. Does the registered `<|endoftext|>` token receive the expected ID?
2. Does the ByteLevel alphabet really provide all 256 byte symbols?
3. Can text containing punctuation, Unicode, emoji, and newlines round-trip without an unknown token?
4. Is the boundary/EOS token inserted only when we explicitly supply it?

The probe is **not** the project tokenizer. A tiny corpus cannot learn 16,384 useful vocabulary entries, and we are not yet locking the full-corpus merge-frequency policy. Full WikiText tokenizer training remains the next phase after these mechanics pass.


## 9. Pin the tokenizer implementation

Tokenizer behavior is part of reproducibility, so this notebook now installs and checks Hugging Face `tokenizers==0.23.1`.

The project already owns linguistic normalization in `src/data.py`. The tokenizer therefore does **not** add a second text normalizer here. ByteLevel's job is different: it maps raw UTF-8 bytes into a reversible visible alphabet and applies GPT-style pre-tokenization.


In [ ]:
import tokenizers
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

EXPECTED_TOKENIZERS_VERSION = "0.23.1"

assert tokenizers.__version__ == EXPECTED_TOKENIZERS_VERSION, (
    f"Unexpected tokenizers version: {tokenizers.__version__}"
)

print(f"✓ tokenizers version pinned: {tokenizers.__version__}")


✓ tokenizers version pinned: 0.23.1


## 10. Instantiate the byte-level BPE components

The intended tokenizer has three core pieces:

- `models.BPE()` — the vocabulary starts from symbols and learns frequent pair merges.
- `pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)` — exposes every UTF-8 byte through a 256-symbol reversible alphabet and uses GPT-style splitting behavior.
- `decoders.ByteLevel()` — reverses the byte-to-visible-character mapping during decode.

`add_prefix_space=False` is intentional: we preserve whether the original text actually began with a space rather than silently manufacturing one.

The trainer reserves `<|endoftext|>` first and explicitly seeds the complete 256-symbol ByteLevel alphabet. The probe leaves `min_frequency` at the library default because that is a **full-training policy choice**, not something needed to verify these mechanics.


In [ ]:
def build_byte_bpe_components(show_progress=False):
    tokenizer_object = Tokenizer(models.BPE())
    tokenizer_object.pre_tokenizer = pre_tokenizers.ByteLevel(
        add_prefix_space=False,
        use_regex=True,
    )
    tokenizer_object.decoder = decoders.ByteLevel()

    trainer_object = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=show_progress,
    )
    return tokenizer_object, trainer_object


byte_alphabet = pre_tokenizers.ByteLevel.alphabet()

assert len(byte_alphabet) == BYTE_ALPHABET_SIZE
assert len(set(byte_alphabet)) == BYTE_ALPHABET_SIZE

probe_tokenizer, probe_trainer = build_byte_bpe_components()

print(f"✓ ByteLevel alphabet contains {len(byte_alphabet)} unique symbols.")
print(f"✓ Requested vocabulary ceiling remains {VOCAB_SIZE:,} total IDs.")


✓ ByteLevel alphabet contains 256 unique symbols.
✓ Requested vocabulary ceiling remains 16,384 total IDs.


## 11. Train only a tiny mechanics probe

A BPE model has no usable vocabulary until a trainer observes some text. We therefore train on four short, fixed strings—not on WikiText—to make the configured tokenizer operational.

Because the corpus is tiny, the probe will stop far below 16,384 entries. That is expected. What matters here is **ordering and coverage**, especially the special-token ID and the seeded byte alphabet.


In [ ]:
PROBE_TEXTS = [
    "Hello, tokenizer mechanics.",
    "Byte-level BPE keeps punctuation: 3.5%, 1,000, 12:30.",
    "Unicode stays reversible: café naïve résumé.",
    "Emoji and symbols stay reversible too: 🙂 — £ €.",
]

probe_tokenizer.train_from_iterator(
    PROBE_TEXTS,
    trainer=probe_trainer,
    length=len(PROBE_TEXTS),
)

probe_vocab = probe_tokenizer.get_vocab()
probe_vocab_size = len(probe_vocab)
boundary_token_id = probe_tokenizer.token_to_id(DOC_BOUNDARY_TOKEN)
byte_symbol_ids = {
    symbol: probe_tokenizer.token_to_id(symbol)
    for symbol in byte_alphabet
}

assert boundary_token_id is not None
assert boundary_token_id == 0
assert all(token_id is not None for token_id in byte_symbol_ids.values())
assert probe_vocab_size >= 1 + BYTE_ALPHABET_SIZE

print(f"Observed boundary token ID: {boundary_token_id}")
print(f"Byte symbols present: {sum(v is not None for v in byte_symbol_ids.values())}/256")
print(f"Probe vocabulary size: {probe_vocab_size:,}")
print(f"Requested full-training vocabulary size: {VOCAB_SIZE:,}")
print("✓ Special-token ordering and complete byte coverage verified.")


Observed boundary token ID: 0
Byte symbols present: 256/256
Probe vocabulary size: 368
Requested full-training vocabulary size: 16,384
✓ Special-token ordering and complete byte coverage verified.


## 12. Verify byte-level reversibility

Byte-level tokenization is useful because unseen Unicode characters do not require an `<unk>` token. UTF-8 first represents a character as one or more bytes; the ByteLevel alphabet guarantees that every possible byte value already has a symbol.

The exact number of tokens below is **not** a quality metric—the probe has learned only a few toy merges. The important check is that encode → decode reconstructs each input exactly and that normal text does not acquire a boundary token automatically.


In [ ]:
ROUNDTRIP_SAMPLES = [
    "Hello, world!",
    "café naïve résumé",
    "🙂 — £ €",
    "line one\nline two",
    " leading space",
]

roundtrip_rows = []

for text in ROUNDTRIP_SAMPLES:
    encoding = probe_tokenizer.encode(text)
    decoded = probe_tokenizer.decode(
        encoding.ids,
        skip_special_tokens=False,
    )

    assert decoded == text
    assert boundary_token_id not in encoding.ids

    roundtrip_rows.append(
        {
            "text": repr(text),
            "utf8_bytes": len(text.encode("utf-8")),
            "probe_tokens": len(encoding.ids),
            "tokens": encoding.tokens,
            "round_trip_exact": decoded == text,
            "boundary_auto_inserted": boundary_token_id in encoding.ids,
        }
    )

print("✓ All byte-level round-trip checks passed.")
pd.DataFrame(roundtrip_rows)


✓ All byte-level round-trip checks passed.


,text,utf8_bytes,probe_tokens,tokens,round_trip_exact,boundary_auto_inserted
0,"'Hello, world!'",13,9,"[Hello, ,, Ġ, w, o, r, l, d, !]",True,False
1,'café naïve résumé',21,4,"[caf, Ã©, ĠnaÃ¯ve, ĠrÃ©sumÃ©]",True,False
2,'🙂 — £ €',15,7,"[ðŁ, ĻĤ, ĠâĢĶ, ĠÂ£, Ġâ, Ĥ, ¬]",True,False
3,'line one\nline two',17,17,"[l, i, n, e, Ġ, o, n, e, Ċ, l, i, n, e, Ġ, t, ...",True,False
4,' leading space',14,13,"[Ġ, l, e, a, d, i, n, g, Ġs, p, a, c, e]",True,False


## 13. Prove that EOS is structural, atomic, and explicit

Registering `<|endoftext|>` gives that literal string one reserved token ID. It does **not** create a post-processor that appends EOS to every encoded sequence.

So there are two distinct operations:

- encode ordinary document text → **no boundary token appears**;
- explicitly include/append `<|endoftext|>` at a document boundary → the tokenizer emits the reserved ID exactly once.

That distinction is what lets the later corpus-construction code control article boundaries precisely and count them inside the exact 20M-token budget.


In [ ]:
plain_text = "Document body."
explicit_boundary_text = plain_text + DOC_BOUNDARY_TOKEN

plain_encoding = probe_tokenizer.encode(plain_text)
explicit_encoding = probe_tokenizer.encode(explicit_boundary_text)

assert boundary_token_id not in plain_encoding.ids
assert explicit_encoding.ids.count(boundary_token_id) == 1
assert explicit_encoding.tokens.count(DOC_BOUNDARY_TOKEN) == 1

special_behavior = pd.Series(
    {
        "boundary_token": DOC_BOUNDARY_TOKEN,
        "boundary_token_id": boundary_token_id,
        "plain_ids_contain_boundary": boundary_token_id in plain_encoding.ids,
        "explicit_boundary_count": explicit_encoding.ids.count(boundary_token_id),
        "decode_keep_special": probe_tokenizer.decode(
            explicit_encoding.ids,
            skip_special_tokens=False,
        ),
        "decode_skip_special": probe_tokenizer.decode(
            explicit_encoding.ids,
            skip_special_tokens=True,
        ),
    }
)

print("✓ Boundary token is explicit rather than automatically inserted.")
special_behavior


✓ Boundary token is explicit rather than automatically inserted.


,0
boundary_token,<|endoftext|>
boundary_token_id,0
plain_ids_contain_boundary,False
explicit_boundary_count,1
decode_keep_special,Document body.<|endoftext|>
decode_skip_special,Document body.


### Pause here

Chunk 3 is a mechanics gate. **Do not train the full WikiText tokenizer yet.**

Before the full run, the outputs should establish:

- `tokenizers == 0.23.1`;
- `<|endoftext|>` has observed token ID **0** under the intended trainer ordering;
- all **256/256** ByteLevel symbols are present;
- punctuation, Unicode, emoji, newlines, and leading spaces round-trip exactly;
- ordinary text receives **no automatic boundary token**; and
- an explicitly supplied `<|endoftext|>` is recognized atomically exactly once.

A probe vocabulary smaller than 16,384 is expected because four toy strings cannot supply enough distinct useful merges.

**Next reviewed chunk:** lock the remaining full-training BPE policy (especially merge-frequency handling), then train the actual tokenizer on the complete normalized training split.
